# Algorithmic Trading and Quantitative Strategies
## Part 3: Financial Data Analysis and Stylized Facts
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. [Loading Financial Data](#1.-Loading-Financial-Data)
2. [Stylized Facts on Asset Returns](#2.-Stylized-Facts-on-Asset-Returns)
3. [Descriptive Statistics](#3.-Descriptive-Statistics)
4. [Correlation Analysis](#4.-Correlation-Analysis)
5. [Exercises](#5.-Exercises)

## 1. Loading Financial Data

In this notebook, we will explore the statistical properties of financial returns — known as *stylized facts*. These empirical regularities are important to understand before designing any trading strategy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

### 1.1 Downloading Data

We use `yfinance` to download OHLC data for several assets.

In [ ]:
# Download data for analysis
tickers = {
    'SPY': 'S&P 500 ETF',
    'EEM': 'Emerging Markets ETF',
    'GLD': 'Gold ETF',
    'TLT': 'US Treasury Bond ETF',
}

data = yf.download(list(tickers.keys()), start='2005-01-01', end='2024-12-31')
close = data['Close']
close.columns = close.columns.droplevel(1) if isinstance(close.columns, pd.MultiIndex) else close.columns
print(f"Data range: {close.index[0].date()} to {close.index[-1].date()}")
print(f"Shape: {close.shape}")
close.tail()

### 1.2 Calculating Returns

In [ ]:
# Simple returns
simple_returns = close.pct_change().dropna()

# Log returns
log_returns = np.log(close / close.shift(1)).dropna()

print("Simple returns shape:", simple_returns.shape)
print("\nFirst few rows:")
simple_returns.head()

### 1.3 Price and Return Plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Normalized prices
normalized = close / close.iloc[0] * 100
for col in normalized.columns:
    axes[0].plot(normalized[col], label=f"{col} ({tickers[col]})")
axes[0].set_title('Normalized Prices (Base = 100)')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Returns
for col in simple_returns.columns:
    axes[1].plot(simple_returns[col], alpha=0.5, label=col, linewidth=0.5)
axes[1].set_title('Daily Returns')
axes[1].set_ylabel('Return')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Stylized Facts on Asset Returns

Asset returns across different markets and time periods share a number of common statistical properties, known as **stylized facts**. Understanding these is crucial for strategy design.

### Stylized Fact 1: Returns are approximately uncorrelated

Daily returns show little to no serial autocorrelation, consistent with (near) market efficiency.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

symbol = 'SPY'
ret = simple_returns[symbol]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ret, lags=30, ax=axes[0], title=f'ACF of {symbol} Returns')
plot_acf(ret**2, lags=30, ax=axes[1], title=f'ACF of {symbol} Squared Returns')
plt.tight_layout()
plt.show()

# Ljung-Box test for autocorrelation
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(ret, lags=10, return_df=True)
print("Ljung-Box Test on Returns:")
print(lb_test)

### Stylized Fact 2: Fat Tails (Leptokurtosis)

Return distributions have heavier tails than the normal distribution. This means extreme events (crashes, spikes) occur more frequently than a Gaussian model would predict.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram vs Normal
ret_std = (ret - ret.mean()) / ret.std()
x = np.linspace(-5, 5, 200)
axes[0].hist(ret_std, bins=100, density=True, alpha=0.7, edgecolor='black', label='Empirical')
axes[0].plot(x, stats.norm.pdf(x), 'r-', linewidth=2, label='Normal')
axes[0].plot(x, stats.t.pdf(x, df=5), 'g--', linewidth=2, label='Student-t (df=5)')
axes[0].set_title(f'{symbol} Standardized Returns vs Normal')
axes[0].set_xlim(-5, 5)
axes[0].legend()

# QQ Plot
stats.probplot(ret, dist="norm", plot=axes[1])
axes[1].set_title(f'QQ Plot: {symbol} Returns vs Normal')

plt.tight_layout()
plt.show()

# Kurtosis comparison
for col in simple_returns.columns:
    r = simple_returns[col]
    print(f"{col}: Skewness={r.skew():.3f}, Excess Kurtosis={r.kurtosis():.3f}")
print(f"\nNormal distribution: Skewness=0, Excess Kurtosis=0")

### Stylized Fact 3: Volatility Clustering

Large returns (in absolute value) tend to be followed by large returns, and small returns by small returns. Formally, $|r_t|$ and $r_t^2$ exhibit significant positive autocorrelation.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Absolute returns
axes[0].plot(ret.abs(), alpha=0.5, linewidth=0.5)
axes[0].plot(ret.abs().rolling(60).mean(), 'r-', linewidth=1.5, label='60-day MA of |Returns|')
axes[0].set_title(f'{symbol} Absolute Returns (Volatility Clustering)')
axes[0].legend()

# Rolling volatility
rolling_vol = ret.rolling(20).std() * np.sqrt(252)
axes[1].plot(rolling_vol, 'b-', linewidth=1)
axes[1].set_title(f'{symbol} Rolling 20-day Annualized Volatility')
axes[1].set_ylabel('Volatility')

plt.tight_layout()
plt.show()

### Stylized Fact 4: Leverage/Asymmetry Effect

Negative returns tend to increase volatility more than positive returns of the same magnitude. This is sometimes called the *leverage effect*.

In [ ]:
# Compare volatility after positive vs negative returns
pos_mask = ret > 0
neg_mask = ret < 0

# Next-day absolute return after positive vs negative days
next_abs_ret = ret.abs().shift(-1)
avg_after_pos = next_abs_ret[pos_mask].mean()
avg_after_neg = next_abs_ret[neg_mask].mean()

print(f"Avg |return| after positive day: {avg_after_pos:.6f}")
print(f"Avg |return| after negative day: {avg_after_neg:.6f}")
print(f"Ratio (neg/pos): {avg_after_neg/avg_after_pos:.3f}")

### Stylized Fact 5: Volume-Volatility Correlation

Trading volume and volatility tend to be positively correlated.

In [ ]:
spy_data = yf.download('SPY', start='2020-01-01', end='2024-12-31')
spy_data.columns = spy_data.columns.droplevel('Ticker')
spy_ret = spy_data['Close'].pct_change().dropna()
spy_vol = spy_data['Volume'].iloc[1:]  # align with returns

corr = spy_ret.abs().corr(spy_vol)
print(f"Correlation between |returns| and volume: {corr:.4f}")

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(spy_ret.abs().rolling(20).mean(), 'b-', alpha=0.7, label='|Returns| (20d MA)')
ax1.set_ylabel('Absolute Return', color='b')
ax2 = ax1.twinx()
ax2.plot(spy_vol.rolling(20).mean(), 'r-', alpha=0.5, label='Volume (20d MA)')
ax2.set_ylabel('Volume', color='r')
ax1.set_title('SPY: Volatility and Volume')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.show()

## 3. Descriptive Statistics

Let us compute comprehensive statistics for our asset universe.

In [ ]:
def compute_stats(returns, freq=252):
    """Compute common descriptive statistics for a return series."""
    stats_dict = {
        'Observations': len(returns),
        'Ann. Return': returns.mean() * freq,
        'Ann. Volatility': returns.std() * np.sqrt(freq),
        'Sharpe Ratio': returns.mean() / returns.std() * np.sqrt(freq),
        'Skewness': returns.skew(),
        'Excess Kurtosis': returns.kurtosis(),
        'Min Daily': returns.min(),
        'Max Daily': returns.max(),
        'Max Drawdown': ((1 + returns).cumprod() / (1 + returns).cumprod().cummax() - 1).min(),
    }
    return stats_dict

# Compute for all assets
stats_table = pd.DataFrame({col: compute_stats(simple_returns[col]) for col in simple_returns.columns})
print(stats_table.round(4).to_string())

### Drawdown Analysis

In [ ]:
# Drawdown calculation and plot
cumulative = (1 + simple_returns).cumprod()
running_max = cumulative.cummax()
drawdown = cumulative / running_max - 1

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for col in cumulative.columns:
    axes[0].plot(cumulative[col], label=col)
axes[0].set_title('Cumulative Returns')
axes[0].legend()
axes[0].set_ylabel('Growth of $1')

for col in drawdown.columns:
    axes[1].fill_between(drawdown.index, drawdown[col], 0, alpha=0.3, label=col)
axes[1].set_title('Drawdowns')
axes[1].legend()
axes[1].set_ylabel('Drawdown')

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation matrix
corr = simple_returns.corr()
print("Correlation Matrix:")
print(corr.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=12)
fig.colorbar(im)
ax.set_title('Return Correlation Matrix')
plt.tight_layout()
plt.show()

### Rolling Correlation

In [ ]:
# Rolling 60-day correlation between SPY and other assets
fig, ax = plt.subplots(figsize=(14, 5))
for col in ['EEM', 'GLD', 'TLT']:
    rolling_corr = simple_returns['SPY'].rolling(60).corr(simple_returns[col])
    ax.plot(rolling_corr, label=f'SPY vs {col}', linewidth=1)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.set_title('Rolling 60-day Correlation with SPY')
ax.set_ylabel('Correlation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Exercises

1. Download daily data for the BIST100 index (ticker: `XU100.IS`) from 2010 to 2024 using yfinance.
2. Calculate daily log returns and compute annualized return and volatility.
3. Test for normality using the Jarque-Bera test (`scipy.stats.jarque_bera`).
4. Plot the return distribution against a fitted normal and Student-t distribution.
5. Compute and plot the rolling 20-day volatility. Identify the highest-volatility periods.
6. Compare the autocorrelation of returns vs. absolute returns. What does this tell you about market efficiency and volatility clustering?

---
### References
- Cont, R. (2001). *Empirical properties of asset returns: stylized facts and statistical issues.* Quantitative Finance, 1(2), 223-236.
- Campbell, J.Y., Lo, A.W., & MacKinlay, A.C. (1997). *The Econometrics of Financial Markets.* Princeton University Press.